# Drone Object Detection — Full Training

Run this notebook **after** completing `train.ipynb` (subset comparison) and reviewing
results in `colab/results/analyze.ipynb`.

**Steps:**
1. Set `BEST_MODEL` in Cell 1 to the winning architecture name (e.g. `"yolov8s"`)
2. Run all cells top to bottom
3. The final trained weights and test-set evaluation are saved to Drive under
   `drone_detection_training/final_model/`

Valid values for `BEST_MODEL`: `yolov8n`, `yolov8s`, `yolov8m`, `yolov8n_tuned`

In [ ]:
# -- Cell 1: Configuration ---------------------------------------------------
# Set BEST_MODEL to whichever architecture won in the subset comparison.

BEST_MODEL  = 'yolov8m'   # <-- CHANGE THIS after reviewing analyze.ipynb

DRIVE_BASE  = '/content/drive/MyDrive/drone_detection_training'
DATASET_ZIP = 'combined_yolo.zip'
EPOCHS      = 100
IMG_SIZE    = 640
SEED        = 42

# Hyperparameters per architecture (must match the comparison run for fairness)
_MODEL_CONFIGS = {
    'yolov8n': {
        'weights': 'yolov8n.pt',
        'batch':   16,
    },
    'yolov8s': {
        'weights': 'yolov8s.pt',
        'batch':   12,
    },
    'yolov8m': {
        'weights': 'yolov8m.pt',
        'batch':   8,
    },
    'yolov8n_tuned': {
        'weights': 'yolov8n.pt',
        'batch':   16,
        'lr0':     0.001,
        'lrf':     0.01,
        'mosaic':  1.0,
        'hsv_h':   0.015,
        'hsv_s':   0.7,
        'hsv_v':   0.4,
        'flipud':  0.1,
        'fliplr':  0.5,
    },
}

if BEST_MODEL not in _MODEL_CONFIGS:
    raise ValueError(f'Unknown model "{BEST_MODEL}". Valid options: {list(_MODEL_CONFIGS)}')

MODEL_CFG = _MODEL_CONFIGS[BEST_MODEL]
print(f'Model:      {BEST_MODEL}')
print(f'Weights:    {MODEL_CFG["weights"]}')
print(f'Epochs:     {EPOCHS}')
print(f'Batch size: {MODEL_CFG["batch"]}')
print(f'Drive base: {DRIVE_BASE}')

Model:      yolov8m
Weights:    yolov8m.pt
Epochs:     100
Batch size: 8
Drive base: /content/drive/MyDrive/drone_detection_training


In [ ]:
# -- Cell 2: Install Dependencies --------------------------------------------
import subprocess, sys

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'ultralytics>=8.0.0', 'torch', 'torchvision'],
    check=True
)

import torch

if torch.cuda.is_available():
    device = torch.cuda.get_device_name(0)
    mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f'GPU: {device}  |  VRAM: {mem_gb:.1f} GB')
else:
    print('WARNING: No GPU detected. Training will be very slow on CPU.')
    print('Go to Runtime -> Change runtime type -> GPU')

GPU: NVIDIA A100-SXM4-40GB  |  VRAM: 39.5 GB


In [ ]:
# -- Cell 3: Mount Google Drive ----------------------------------------------
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

base      = Path(DRIVE_BASE)
final_dir = base / 'final_model'
final_dir.mkdir(parents=True, exist_ok=True)

print(f'Drive folder: {base}')
print(f'Output dir:   {final_dir}')

Mounted at /content/drive
Drive folder: /content/drive/MyDrive/drone_detection_training
Output dir:   /content/drive/MyDrive/drone_detection_training/final_model


In [ ]:
# -- Cell 4: Dataset Integrity Check (Full Dataset) --------------------------
import shutil
from pathlib import Path

DATASET_DIR = Path('/content/dataset')
zip_path    = Path(DRIVE_BASE) / 'datasets' / DATASET_ZIP

if not zip_path.exists():
    print('ERROR: Dataset zip not found.')
    print(f'  Expected: {zip_path}')
    raise SystemExit(1)

if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
DATASET_DIR.mkdir(parents=True)

print(f'Unzipping {DATASET_ZIP} to {DATASET_DIR} ...')
shutil.unpack_archive(str(zip_path), str(DATASET_DIR))
print('Unzip complete.')

errors = {}
counts = {}

for split in ('train', 'val', 'test'):
    img_dir = DATASET_DIR / 'images' / split
    lbl_dir = DATASET_DIR / 'labels' / split

    if not img_dir.exists():
        errors[split] = f'Missing: images/{split}/'
        continue
    if not lbl_dir.exists():
        errors[split] = f'Missing: labels/{split}/'
        continue

    imgs = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.jpeg')) + list(img_dir.glob('*.png'))
    lbls = list(lbl_dir.glob('*.txt'))
    counts[split] = {'images': len(imgs), 'labels': len(lbls)}

    if len(imgs) != len(lbls):
        errors[split] = f'{split}: {len(imgs)} images vs {len(lbls)} labels (mismatch)'

yaml_files = list(DATASET_DIR.glob('*.yaml'))
if not yaml_files:
    errors['yaml'] = 'No .yaml file found in dataset root'
else:
    DATASET_YAML = str(yaml_files[0])

print()
print(f'{"Split":<8} {"Images":>8} {"Labels":>8} {"Match":>7}')
print('-' * 35)
for split, c in counts.items():
    match = 'OK' if c['images'] == c['labels'] else 'MISMATCH'
    print(f'{split:<8} {c["images"]:>8} {c["labels"]:>8} {match:>7}')

if yaml_files:
    print(f'\nDataset yaml: {yaml_files[0].name}')

if errors:
    print('\nERRORS:')
    for e in errors.values():
        print(f'  - {e}')
    raise SystemExit(1)

print('\nDataset integrity check passed.')

Unzipping combined_yolo.zip to /content/dataset ...
Unzip complete.

Split      Images   Labels   Match
-----------------------------------
train        7737     7737      OK
val           819      819      OK
test         1882     1882      OK

Dataset yaml: combined.yaml

Dataset integrity check passed.


In [ ]:
# -- Cell 5: Full Training ---------------------------------------------------
import json
import time
import shutil
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

drive_out = Path(DRIVE_BASE) / 'final_model'

if (drive_out / 'results.csv').exists():
    print(f'[SKIP] Final model already trained. Delete {drive_out} to retrain.')
else:
    print(f'Training {BEST_MODEL} on full dataset for {EPOCHS} epochs ...')
    print(f'Dataset yaml: {DATASET_YAML}')
    print()

    model        = YOLO(MODEL_CFG['weights'])
    train_kwargs = {k: v for k, v in MODEL_CFG.items() if k != 'weights'}

    t_start = time.time()
    model.train(
        data=DATASET_YAML,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        project='/content/runs',
        name='final_model',
        exist_ok=False,
        seed=SEED,
        **train_kwargs,
    )
    training_time = time.time() - t_start

    run_dir       = Path('/content/runs/final_model')
    best_pt       = run_dir / 'weights' / 'best.pt'
    model_size_mb = round(best_pt.stat().st_size / 1e6, 2) if best_pt.exists() else None

    df = pd.read_csv(run_dir / 'results.csv')
    df.columns = df.columns.str.strip()
    best_idx = df['metrics/mAP50(B)'].idxmax()

    run_info = {
        'model_name':            BEST_MODEL,
        'weights':               MODEL_CFG['weights'],
        'dataset':               'combined (full)',
        'epochs_completed':      len(df),
        'best_epoch':            int(best_idx) + 1,
        'training_time_seconds': round(training_time, 1),
        'model_size_mb':         model_size_mb,
        'config':                {k: v for k, v in MODEL_CFG.items() if k != 'weights'},
        'best_metrics': {
            'mAP50':     round(float(df.loc[best_idx, 'metrics/mAP50(B)']),     4),
            'mAP50_95':  round(float(df.loc[best_idx, 'metrics/mAP50-95(B)']), 4),
            'precision': round(float(df.loc[best_idx, 'metrics/precision(B)']), 4),
            'recall':    round(float(df.loc[best_idx, 'metrics/recall(B)']),    4),
        },
    }
    with open(run_dir / 'run_info.json', 'w') as fh:
        json.dump(run_info, fh, indent=2)

    # Atomic copy to Drive
    tmp_dst = drive_out.parent / 'final_model_tmp'
    if tmp_dst.exists():
        shutil.rmtree(tmp_dst)

    (tmp_dst / 'weights').mkdir(parents=True, exist_ok=True)
    for wf in (run_dir / 'weights').glob('*.pt'):
        shutil.copy2(str(wf), str(tmp_dst / 'weights' / wf.name))

    for item in run_dir.rglob('*'):
        if item.is_file() and 'weights' not in item.parts:
            rel    = item.relative_to(run_dir)
            target = tmp_dst / rel
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(str(item), str(target))

    if drive_out.exists():
        shutil.rmtree(drive_out)
    tmp_dst.rename(drive_out)

    bm    = run_info['best_metrics']
    t_min = training_time / 60
    print(f'\n[DONE] {BEST_MODEL} full training complete')
    print(f'       mAP50={bm["mAP50"]}  mAP50-95={bm["mAP50_95"]}')
    print(f'       time={t_min:.1f}min  size={model_size_mb}MB')
    print(f'       Saved to: {drive_out}')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Training yolov8m on full dataset for 100 epochs ...
Dataset yaml: /content/dataset/combined.yaml

Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/combined.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=

In [ ]:
# -- Cell 6: Held-Out Test Set Evaluation ------------------------------------
# Runs the best checkpoint against the test split and saves test metrics.
# Only run after training completes.

import json
from pathlib import Path
from ultralytics import YOLO

best_weights = Path(DRIVE_BASE) / 'final_model' / 'weights' / 'best.pt'
if not best_weights.exists():
    print('ERROR: best.pt not found. Run the training cell first.')
    raise SystemExit(1)

print(f'Evaluating {best_weights} on test split ...')
model   = YOLO(str(best_weights))
results = model.val(
    data=DATASET_YAML,
    split='test',
    imgsz=IMG_SIZE,
    batch=MODEL_CFG['batch'],
)

test_metrics = {
    'model_name': BEST_MODEL,
    'split':      'test',
    'mAP50':      round(float(results.box.map50),  4),
    'mAP50_95':   round(float(results.box.map),    4),
    'precision':  round(float(results.box.mp),     4),
    'recall':     round(float(results.box.mr),     4),
}

test_results_path = Path(DRIVE_BASE) / 'final_model' / 'test_results.json'
with open(test_results_path, 'w') as fh:
    json.dump(test_metrics, fh, indent=2)

print()
print('Test Set Results')
print('=' * 40)
for k, v in test_metrics.items():
    print(f'  {k:<12}: {v}')
print(f'\nSaved to: {test_results_path}')

Evaluating /content/drive/MyDrive/drone_detection_training/final_model/weights/best.pt on test split ...
Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
Model summary (fused): 93 layers, 25,841,497 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2266.3±429.0 MB/s, size: 166.3 KB)
val: Scanning /content/dataset/labels/test... 1882 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1882/1882 1.3Kit/s 1.4s
val: New cache created: /content/dataset/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 236/236 13.2it/s 17.9s
                   all       1882      83059      0.678      0.504      0.515       0.27
                person       1267      27382      0.611      0.302      0.318      0.123
               vehicle       1836      47401      0.859      0.797       0.83      0.522
           two_wheeler        903       8276    

In [ ]:
# -- Cell 7: Final Summary ---------------------------------------------------
import json
from pathlib import Path

final_dir = Path(DRIVE_BASE) / 'final_model'

run_info_path  = final_dir / 'run_info.json'
test_res_path  = final_dir / 'test_results.json'

if run_info_path.exists():
    with open(run_info_path) as fh:
        info = json.load(fh)
    bm = info['best_metrics']
    t_min = info['training_time_seconds'] / 60
    print('Full Training Summary')
    print('=' * 50)
    print(f'  Model:          {info["model_name"]}')
    print(f'  Dataset:        {info["dataset"]}')
    print(f'  Epochs:         {info["epochs_completed"]}  (best: {info["best_epoch"]})')
    print(f'  Training time:  {t_min:.1f} min')
    print(f'  Model size:     {info["model_size_mb"]} MB')
    print()
    print('  Val metrics (best epoch):')
    print(f'    mAP50:     {bm["mAP50"]}')
    print(f'    mAP50-95:  {bm["mAP50_95"]}')
    print(f'    Precision: {bm["precision"]}')
    print(f'    Recall:    {bm["recall"]}')

if test_res_path.exists():
    with open(test_res_path) as fh:
        tm = json.load(fh)
    print()
    print('  Test set metrics:')
    print(f'    mAP50:     {tm["mAP50"]}')
    print(f'    mAP50-95:  {tm["mAP50_95"]}')
    print(f'    Precision: {tm["precision"]}')
    print(f'    Recall:    {tm["recall"]}')

print()
print(f'Final weights: {final_dir}/weights/best.pt')
print(f'Drive folder:  {final_dir}')

Full Training Summary
  Model:          yolov8m
  Dataset:        combined (full)
  Epochs:         100  (best: 77)
  Training time:  194.5 min
  Model size:     52.04 MB

  Val metrics (best epoch):
    mAP50:     0.6438
    mAP50-95:  0.3467
    Precision: 0.7428
    Recall:    0.6094

  Test set metrics:
    mAP50:     0.5154
    mAP50-95:  0.2704
    Precision: 0.6783
    Recall:    0.5041

Final weights: /content/drive/MyDrive/drone_detection_training/final_model/weights/best.pt
Drive folder:  /content/drive/MyDrive/drone_detection_training/final_model
